In [ ]:
from datasets import load_from_disk
from scripts.vllm_vision_engine import PhenoVisionClient
import torch, json
from scripts.helpers import build_emb_model_for_vision
from scripts.utils import map_vision_to_hpo

emb_model = build_emb_model_for_vision()

# Load the test split
ds = load_from_disk("/home/nguyenqm/projects/github/PhenoGPT2/vision_gmdb_data_new/phenotype_only")["test"]
#model_name = 'vision_qwen3_8b_vl_phenotype_mixed_nostyle'
model_name = '/home/nguyenqm/projects/github/PhenoGPT2/vision_qwen3_8b_vl_phenotype_only_one_strip/model'
# Initialize vision client
vision_client = PhenoVisionClient(
    #model_path=f"/home/nguyenqm/projects/github_official/PhenoGPT2/models/{model_name}/model",
    model_path=model_name,
    gpu_memory_utilization=0.85,
    tensor_parallel_size=torch.cuda.device_count(),
    max_batch_size=15
)

In [ ]:
from sentence_transformers import SentenceTransformer, util

def evaluate_hpo_matching(pred_list, golden_list, emb_model, threshold=0.8):
    """
    Match predicted HPO terms to golden HPO terms using embedding similarity.
    Returns matched pairs, precision, recall, and F1 score.
    """
    if len(golden_list) == 0:
        if len(pred_list) == 0:
            return {
                "matched_pairs": [],
                "num_matched": 1,
                "precision": 1,
                "recall": 1,
                "f1": 1,
            }
        else:
            return {
                "matched_pairs": [],
                "num_matched": 0,
                "precision": 0,
                "recall": 0,
                "f1": 0,
            }
    pred_embs = emb_model.encode(pred_list, convert_to_tensor=True,  show_progress_bar=False)
    golden_embs = emb_model.encode(golden_list, convert_to_tensor=True, show_progress_bar=False)
    
    sim_matrix = util.cos_sim(pred_embs, golden_embs)  # (len(pred), len(golden))
    
    matched_preds = set()
    matched_goldens = set()
    matched_pairs = []

    for i in range(len(pred_list)):
        for j in range(len(golden_list)):
            if sim_matrix[i][j] >= threshold and i not in matched_preds and j not in matched_goldens:
                matched_preds.add(i)
                matched_goldens.add(j)
                matched_pairs.append((pred_list[i], golden_list[j], float(sim_matrix[i][j])))

    tp = len(matched_pairs)
    precision = tp / len(pred_list) if pred_list else 0.0
    recall = tp / len(golden_list) if golden_list else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "matched_pairs": matched_pairs,
        "num_matched": tp,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
# Run batched inference
outputs = vision_client.generate_from_dataset(ds)  # list[str] aligned with ds
del vision_client
# embedding-based HPO mapping (no text model needed)
hpo_mapped_list = map_vision_to_hpo(
    outputs, emb_model, threshold=0.8
)